# Concurrency and parallelism
* until now our code always ran one instruction after another, top to bottom, on a single cpu core - "sequential" code
* real programs often have to WAIT a lot (for a network response, a disk read, a database, user input) - during that wait, the cpu is basically idle and bored
* other programs need to CRUNCH a lot of numbers (image processing, math, compression) - during that the cpu is fully busy
* this lesson is about doing multiple things "at once" to not waste that idle time or to actually speed up crunching - python gives us several tools for this, each with its own tradeoffs
* two words that sound similar but mean different things:
    * **concurrency** = dealing with multiple things "in progress" at the same time (they can overlap, doesnt necessarily mean they run at the exact same instant) - great for waiting-heavy (I/O-bound) work
    * **parallelism** = actually doing multiple things at the exact same instant, which needs multiple cpu cores - great for crunching-heavy (CPU-bound) work
* keep this distinction in mind, its the key to picking the right tool later in this lesson

## the GIL (Global Interpreter Lock)
* CPython (the "normal" python interpreter we all use) has an internal lock called the GIL
* the GIL only allows ONE thread to execute python bytecode at any given moment, no matter how many cpu cores your machine has
* why does this exist? it massively simplifies memory management internally (reference counting) and makes single-threaded code fast and safe - the tradeoff is multi-threaded CPU-bound code
* practical consequence:
    * threads ARE useful for I/O-bound work (waiting for network/disk) - while one thread waits, python happily lets another thread run, the GIL gets released during the wait
    * threads are basically USELESS for speeding up pure-python CPU-bound work - only one thread ever executes python code at a time, so 4 threads doing math dont run 4x faster, they just take turns
* lets demonstrate this idea below - this is NOT a rigorous scientific benchmark (a shared sandbox machine has all kinds of noise), just treat the printed numbers as "roughly showing the idea", not as a guaranteed speedup

In [1]:
import time
import threading

# a deliberately CPU-heavy pure-python function (no I/O, no waiting, just math)
# this is the kind of work the GIL makes hard to parallelize with threads
def cpu_bound_work(n):
    total = 0
    for i in range(n):
        total += i * i  # pure python bytecode, cpu-bound, no waiting involved at all
    return total

work_size = 3_000_000  # kept small on purpose so this cell finishes quickly

# baseline: do the SAME total amount of work sequentially (two chunks, one after another)
start = time.perf_counter()
cpu_bound_work(work_size)
cpu_bound_work(work_size)
sequential_time = time.perf_counter() - start
print(f"sequential (no threads): {sequential_time:.2f}s  <-- baseline, one after another")

# now the same total work, but split across two threads running "at the same time"
start = time.perf_counter()
thread_a = threading.Thread(target=cpu_bound_work, args=(work_size,))  # thread just calls our function with n=work_size
thread_b = threading.Thread(target=cpu_bound_work, args=(work_size,))
thread_a.start()  # start() launches the thread, it begins running concurrently with the main program
thread_b.start()
thread_a.join()   # join() blocks here until the thread has finished, so we can measure the total time fairly
thread_b.join()
threaded_time = time.perf_counter() - start
print(f"two threads (cpu-bound):  {threaded_time:.2f}s  <-- because of the GIL, this is usually NOT much faster, sometimes even slower!")

# the takeaway: for pure CPU-bound python work, threads dont give us real parallelism - the GIL only lets one of them run python bytecode at a time
# again: dont read too much into the exact numbers on a shared sandbox machine, the point is "no dramatic speedup", not an exact ratio

sequential (no threads): 0.24s  <-- baseline, one after another


two threads (cpu-bound):  0.23s  <-- because of the GIL, this is usually NOT much faster, sometimes even slower!


## threading basics
* the `threading` module is pythons classic, low-level way to run things concurrently
* a `threading.Thread` wraps a function and runs it "in the background", concurrently with the rest of your program
* `.start()` kicks the thread off, `.join()` waits (blocks) until it has finished - very similar vocabulary to how we manage processes in an operating system

In [2]:
import threading
import time

# a small function that simulates "waiting" (like a network call) instead of crunching numbers
# this is I/O-bound work - exactly the case where threads shine, because during time.sleep() the GIL gets released
def worker(name, delay):
    print(f"{name}: starting, will wait {delay}s")
    time.sleep(delay)  # simulates waiting for something external (network, disk, ...) - the thread is idle here, not burning cpu
    print(f"{name}: done waiting")

start = time.perf_counter()

# create a few Thread objects - each one gets a target function and the arguments to call it with
threads = []
for i in range(3):
    t = threading.Thread(target=worker, args=(f"worker-{i}", 0.3))  # args must be a tuple, even with just one element
    threads.append(t)
    t.start()  # start() returns immediately, it does NOT wait for the thread to finish - that is the whole point!

# join() blocks the main thread until the given thread has actually finished
for t in threads:
    t.join()

elapsed = time.perf_counter() - start
print(f"all workers finished after {elapsed:.2f}s total")
# because all 3 workers waited concurrently, the total time is close to 0.3s, NOT 3 x 0.3s = 0.9s like it would be sequentially
# this is concurrency paying off for I/O-bound work, even though python only has the GIL

worker-0: starting, will wait 0.3s
worker-1: starting, will wait 0.3s
worker-2: starting, will wait 0.3s


worker-0: done waiting
worker-1: done waiting
worker-2: done waiting
all workers finished after 0.30s total


## race conditions - what can go wrong with shared state
* the big danger of threads: they all share the SAME memory (unlike processes, which we will see later)
* if multiple threads read-and-modify the same variable "at the same time" (well, interleaved by the GIL), updates can get lost
* this is called a **race condition** - the final result depends on unlucky timing/interleaving of the threads, its not deterministic
* lets deliberately break a shared counter to see this happen

In [3]:
import threading

# a plain integer shared between all threads - this is the "shared state" that causes trouble
counter = 0
increments_per_thread = 100_000  # each thread will do this many "counter += 1" operations
thread_count = 4

def increment_without_lock():
    global counter  # we need "global" because we reassign counter inside the function (see lesson 04 on scope)
    for _ in range(increments_per_thread):
        counter += 1  # NOT ATOMIC! this is actually "read counter, add 1, write counter back" as 3 separate steps
        # if the GIL switches to another thread right between "read" and "write", an update can get silently lost

threads = [threading.Thread(target=increment_without_lock) for _ in range(thread_count)]
for t in threads:
    t.start()
for t in threads:
    t.join()

expected = increments_per_thread * thread_count
print(f"expected count: {expected}")
print(f"actual count:   {counter}")
print("wrong!" if counter != expected else "happened to be correct this time, run it again - race conditions are not 100% reliable")
# this is the classic race condition: several threads read/modify/write the same variable without coordinating, some updates get lost

expected count: 400000
actual count:   400000
happened to be correct this time, run it again - race conditions are not 100% reliable


## fixing it with threading.Lock
* a `threading.Lock` is like a "one person at a time" bathroom key - only one thread can hold the lock at any moment
* wrap the critical section (the code that reads-and-modifies shared state) in `with lock:` and python guarantees no other thread can be inside that same block at the same time
* this makes the operation effectively atomic again, at the cost of a little bit of overhead/waiting

In [4]:
import threading

counter = 0
increments_per_thread = 100_000
thread_count = 4
lock = threading.Lock()  # one shared lock object, used by every thread

def increment_with_lock():
    global counter
    for _ in range(increments_per_thread):
        with lock:          # acquires the lock, blocks if another thread currently holds it
            counter += 1    # only ONE thread can be inside this block at any given time -> safe, no lost updates
        # the lock is automatically released again at the end of the "with" block, even if an error happened inside

threads = [threading.Thread(target=increment_with_lock) for _ in range(thread_count)]
for t in threads:
    t.start()
for t in threads:
    t.join()

expected = increments_per_thread * thread_count
print(f"expected count: {expected}")
print(f"actual count:   {counter}")
print("correct every single time now, thanks to the lock" if counter == expected else "something is very wrong")

expected count: 400000
actual count:   400000
correct every single time now, thanks to the lock


## concurrent.futures.ThreadPoolExecutor - the higher level way
* manually creating `Thread` objects and juggling `.start()`/`.join()` gets tedious fast, especially with many tasks
* `concurrent.futures.ThreadPoolExecutor` manages a pool of worker threads for us - we just hand it tasks and it takes care of scheduling them
* this is the recommended, modern way to run many short-lived concurrent tasks in real code

In [5]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# a fake "download" function - again I/O-bound (simulated with sleep), the perfect job for threads
def fetch_url(name, delay):
    time.sleep(delay)  # pretend this is a slow network request
    return f"{name} -> downloaded {delay}s worth of data"

urls = [("page-1", 0.2), ("page-2", 0.3), ("page-3", 0.1), ("page-4", 0.2)]

start = time.perf_counter()
# max_workers=4 means up to 4 threads run at the same time - "with" makes sure the pool is properly shut down afterwards
with ThreadPoolExecutor(max_workers=4) as executor:
    # .submit() schedules ONE task and immediately gives us back a "Future" object (a placeholder for the result-to-come)
    futures = [executor.submit(fetch_url, name, delay) for name, delay in urls]

    # as_completed() yields futures as soon as THEY finish, not necessarily in the order we submitted them
    for future in as_completed(futures):
        print("submit() result:", future.result())  # .result() gives us the actual return value (blocks if not done yet, but here it already is)

elapsed = time.perf_counter() - start
print(f"submit()+as_completed() took {elapsed:.2f}s total (roughly the slowest single task, not the sum of all of them)")

submit() result: page-3 -> downloaded 0.1s worth of data


submit() result: page-1 -> downloaded 0.2s worth of data
submit() result: page-4 -> downloaded 0.2s worth of data
submit() result: page-2 -> downloaded 0.3s worth of data
submit()+as_completed() took 0.30s total (roughly the slowest single task, not the sum of all of them)


In [6]:
import time
from concurrent.futures import ThreadPoolExecutor

def fetch_url(name, delay):
    time.sleep(delay)
    return f"{name} -> downloaded {delay}s worth of data"

urls = [("page-1", 0.2), ("page-2", 0.3), ("page-3", 0.1), ("page-4", 0.2)]

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as executor:
    # .map() is the simpler alternative to submit()+as_completed(): give it a function and a list of argument-tuples-ish inputs
    # it runs them all concurrently and returns the results IN THE SAME ORDER we passed the inputs in (unlike as_completed!)
    results = executor.map(lambda pair: fetch_url(*pair), urls)  # lambda unpacks each (name, delay) tuple into fetch_url's two arguments
    for result in results:  # iterating over .map()'s return value blocks until each result (in order) is ready
        print("map() result:", result)

elapsed = time.perf_counter() - start
print(f"map() took {elapsed:.2f}s total")
# both submit()/as_completed() and map() achieve the same concurrency, map() is just more compact when you dont need per-task control

map() result: page-1 -> downloaded 0.2s worth of data
map() result: page-2 -> downloaded 0.3s worth of data
map() result: page-3 -> downloaded 0.1s worth of data
map() result: page-4 -> downloaded 0.2s worth of data
map() took 0.30s total


## multiprocessing / ProcessPoolExecutor - real parallelism
* `threading` never gives us real CPU parallelism in CPython because of the GIL
* `multiprocessing` sidesteps the GIL completely by using multiple separate OS PROCESSES instead of threads - each process gets its own python interpreter, its own GIL, and (this is the important part) its own memory
* this means real parallelism across cpu cores is possible for CPU-bound work - but it comes with real costs too:
    * starting a process is much heavier/slower than starting a thread
    * processes do NOT share memory - passing data between them means copying/pickling it (via `Queue`, `Pipe`, `Manager`, or the return value of a pool task), you cant just share a plain variable like with threads
* `concurrent.futures.ProcessPoolExecutor` is the higher-level equivalent of `ThreadPoolExecutor`, just backed by processes instead of threads
* important sandbox note: spawning genuinely separate OS processes from code running directly inside a notebook cell can be unreliable, depending on the platform and how the notebook kernel itself was started (multiprocessing really wants a normal `.py` script with `if __name__ == "__main__":`) - so we wrap this demo safely and fall back to an explanation if it doesnt work here, instead of crashing the whole notebook

In [7]:
from concurrent.futures import ProcessPoolExecutor

# a plain module-level-ish function so it CAN be pickled and sent to another process (a nested/local function usually cant be)
def square_number(n):
    return n * n

numbers = [1, 2, 3, 4, 5]

try:
    # ProcessPoolExecutor works just like ThreadPoolExecutor from the outside, but under the hood it starts real OS processes
    with ProcessPoolExecutor(max_workers=2) as executor:
        results = list(executor.map(square_number, numbers))
    print("multiprocessing worked in this environment! results:", results)
    print("each of these squarings potentially ran in its own OS process, on its own cpu core - real parallelism, no GIL involved")
except Exception as error:
    # if the sandbox doesnt allow spawning real subprocesses from inside a notebook kernel, we land here instead of crashing
    print("could not actually spawn separate OS processes here:", repr(error))
    print("multiprocessing across real OS processes needs a real .py script entrypoint (if __name__ == '__main__':),")
    print("inside this notebook environment we simulate the concept instead:")
    simulated_results = [square_number(n) for n in numbers]  # same computation, just sequential, to still show the intended result
    print("simulated (sequential) results:", simulated_results)

multiprocessing worked in this environment! results: [1, 4, 9, 16, 25]
each of these squarings potentially ran in its own OS process, on its own cpu core - real parallelism, no GIL involved


## asyncio - cooperative concurrency with async/await
* `asyncio` is a third way to be concurrent, and its a very different model from threads/processes
* it all happens in a SINGLE thread, but a special kind of function (a "coroutine", defined with `async def`) can voluntarily PAUSE itself at every `await` and hand control back to an event loop
* the event loop then runs whatever else is ready, and comes back to our paused coroutine once whatever it was waiting for (`await asyncio.sleep(...)`, a network request, ...) is done
* the key difference to threads: with threads the OS can interrupt/switch a thread at basically any random point (preemptive) - with asyncio a coroutine only ever gives up control at an `await` point, on its own terms (cooperative) - this makes asyncio code easier to reason about (no random race conditions on shared variables like we saw earlier!) but it ALSO means one coroutine that never awaits (e.g. does heavy CPU work) blocks everything else
* jupyter notebook note: normally you'd write `asyncio.run(main())` in a plain python script, but jupyter's kernel already has its OWN event loop running in the background - calling `asyncio.run()` here would crash with "cannot be called from a running event loop"! instead, jupyter lets us just write `await` directly as the top level of a cell, no `asyncio.run()` needed - we use exactly that below

In [8]:
import asyncio
import time

# an async function ("coroutine") simulating a slow download - note "async def" instead of just "def"
async def download_async(name, delay):
    print(f"{name}: starting download")
    await asyncio.sleep(delay)  # this is the pause point: hands control back to the event loop for "delay" seconds
    print(f"{name}: finished download")
    return f"{name} -> {delay}s of data"

downloads = [("file-A", 0.3), ("file-B", 0.2), ("file-C", 0.4)]

# first, lets do them one after another (sequentially) to have something to compare against
start = time.perf_counter()
sequential_results = []
for name, delay in downloads:
    sequential_results.append(await download_async(name, delay))  # top-level "await" works directly in a jupyter cell
sequential_elapsed = time.perf_counter() - start
print(f"sequential awaits took {sequential_elapsed:.2f}s (roughly the SUM of all delays: 0.3+0.2+0.4=0.9)")
print()

# now lets run all three CONCURRENTLY with asyncio.gather() - it schedules all coroutines and waits for all of them together
start = time.perf_counter()
concurrent_results = await asyncio.gather(*[download_async(name, delay) for name, delay in downloads])
concurrent_elapsed = time.perf_counter() - start
print(f"asyncio.gather() took {concurrent_elapsed:.2f}s (roughly the MAX delay: 0.4, since they all overlap)")
print("results:", concurrent_results)
# just like the threading example earlier, this speedup only works because the "work" here is WAITING, not cpu crunching
# asyncio.sleep() cooperatively gives control back during the wait, so all three downloads can be "in flight" at once, single-threaded

file-A: starting download


file-A: finished download
file-B: starting download
file-B: finished download
file-C: starting download


file-C: finished download
sequential awaits took 0.90s (roughly the SUM of all delays: 0.3+0.2+0.4=0.9)

file-A: starting download
file-B: starting download
file-C: starting download
file-B: finished download


file-A: finished download
file-C: finished download
asyncio.gather() took 0.40s (roughly the MAX delay: 0.4, since they all overlap)
results: ['file-A -> 0.3s of data', 'file-B -> 0.2s of data', 'file-C -> 0.4s of data']


## so... which tool do i reach for?
* **I/O-bound + lots of small tasks + want the simplest/fastest option** -> `asyncio` (single thread, very low overhead, but everything you call needs to be "async-aware")
* **I/O-bound + need to call blocking/legacy libraries that dont know about asyncio** -> `threading` / `ThreadPoolExecutor` (works with any normal blocking code, easy to bolt onto existing code)
* **CPU-bound + you actually need real parallel speedup across cores** -> `multiprocessing` / `ProcessPoolExecutor` (real parallelism, but heavier startup cost and no shared memory)
* rule of thumb: is your code mostly *waiting*? threads or asyncio. is your code mostly *crunching*? multiprocessing. and if in doubt, measure with your real workload instead of guessing

## quick comparison to other languages
* pythons GIL is a CPython-specific design decision, its NOT some universal law of programming - Java, C#, C++, Rust and Go all let multiple normal threads execute your code on multiple cores truly in parallel, no GIL-style lock involved
* this is exactly WHY CPU-bound parallel speedups in python typically need `multiprocessing` instead of `threading` - in those other languages, plain threads are already enough for CPU-bound parallelism
* JavaScript/Node.js is single-threaded by design, just like pythons asyncio model - one event loop, cooperative `async`/`await`, no preemption - pythons asyncio will feel very familiar if you know JS async/await, the concepts map almost 1:1
* Go takes yet another approach: lightweight "goroutines" (cheap, `go myFunction()`) and "channels" for communicating between them - the go RUNTIME automatically schedules goroutines across multiple OS threads for you, giving real parallelism without an explicit GIL-style lock and without you manually managing threads/processes
* Rust gives you BOTH real multi-threading (with fearless concurrency - the compiler enforces memory-safety at compile time via its ownership/borrowing rules, so a lot of race conditions become compile errors instead of runtime bugs) AND an async/await model similar to pythons - its considered one of the most powerful but also most complex concurrency setups among mainstream languages

## Exercises
1. start 3 threads that each print their own thread name (you can pass the name as an argument) and then sleep for a short, random-ish amount of time, then join all of them from the main thread
2. reproduce the classic race-condition counter bug from this lesson (no lock, wrong result) and then fix it with a `threading.Lock`
3. use a `ThreadPoolExecutor` to "fetch" 5 fake urls (simulate each with a short `time.sleep()`), print how long the whole batch took in total
4. write two `async def` functions that each `await asyncio.sleep()` for a short time and run them together with `asyncio.gather()`, using top-level `await` in the cell

In [9]:
# TODO: solve exercise 1 here (3 threads printing their name + sleeping, then join)


# TODO: solve exercise 2 here (race condition without lock, then fixed with lock)


# TODO: solve exercise 3 here (ThreadPoolExecutor fetching 5 fake urls)


# TODO: solve exercise 4 here (two async functions gathered together, top-level await)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [10]:
import threading
import time
import random
from concurrent.futures import ThreadPoolExecutor

# sample solution 1 - 3 threads printing their name and sleeping
def print_and_sleep(name):
    duration = random.uniform(0.1, 0.3)  # a short, slightly random sleep so the interleaving looks a bit more "real"
    print(f"{name}: sleeping for {duration:.2f}s")
    time.sleep(duration)
    print(f"{name}: woke up")

threads = [threading.Thread(target=print_and_sleep, args=(f"thread-{i}",)) for i in range(3)]
for t in threads:
    t.start()
for t in threads:
    t.join()  # main thread waits here until all 3 are done
print("all 3 threads finished")
print()

# sample solution 2 - race condition, then fixed with a lock
counter = 0
def bump_no_lock():
    global counter
    for _ in range(50_000):
        counter += 1  # not atomic, can lose updates when threads interleave

threads = [threading.Thread(target=bump_no_lock) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"without lock -> expected 200000, got {counter} (might be wrong)")

counter = 0
lock = threading.Lock()
def bump_with_lock():
    global counter
    for _ in range(50_000):
        with lock:
            counter += 1  # now safe, only one thread inside this block at a time

threads = [threading.Thread(target=bump_with_lock) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"with lock -> expected 200000, got {counter} (always correct)")
print()

# sample solution 3 - ThreadPoolExecutor fetching 5 fake urls
def fetch(name):
    time.sleep(0.15)  # pretend network delay
    return f"{name} done"

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(fetch, [f"url-{i}" for i in range(5)]))
elapsed = time.perf_counter() - start
print("results:", results)
print(f"fetching 5 fake urls concurrently took {elapsed:.2f}s (not 5 x 0.15s = 0.75s)")
print()

# sample solution 4 - two async functions gathered together
import asyncio

async def task_one():
    await asyncio.sleep(0.2)
    return "task_one finished"

async def task_two():
    await asyncio.sleep(0.3)
    return "task_two finished"

start = time.perf_counter()
gathered = await asyncio.gather(task_one(), task_two())  # top-level await, works directly in a jupyter cell
elapsed = time.perf_counter() - start
print("gathered results:", gathered)
print(f"gathering both took {elapsed:.2f}s (roughly the longer of the two: ~0.3s, not the sum ~0.5s)")

thread-0: sleeping for 0.23s
thread-1: sleeping for 0.15s
thread-2: sleeping for 0.11s


thread-2: woke up
thread-1: woke up


thread-0: woke up
all 3 threads finished

without lock -> expected 200000, got 200000 (might be wrong)
with lock -> expected 200000, got 200000 (always correct)



results: ['url-0 done', 'url-1 done', 'url-2 done', 'url-3 done', 'url-4 done']
fetching 5 fake urls concurrently took 0.15s (not 5 x 0.15s = 0.75s)



gathered results: ['task_one finished', 'task_two finished']
gathering both took 0.30s (roughly the longer of the two: ~0.3s, not the sum ~0.5s)


* congratulation you finished this lesson

## what we learned
* the difference between concurrency (dealing with many things at once) and parallelism (doing many things at the exact same instant, needs multiple cores)
* the GIL: why CPython threads dont speed up pure CPU-bound python work, but are great for I/O-bound waiting
* `threading.Thread`, `.start()`/`.join()`, and how a race condition on shared state without a `threading.Lock` can silently lose updates
* `concurrent.futures.ThreadPoolExecutor` as the higher-level, recommended way to run many concurrent I/O-bound tasks (`.submit()`+`as_completed()`, and `.map()`)
* `multiprocessing`/`ProcessPoolExecutor`: real parallelism via separate OS processes that sidestep the GIL, at the cost of startup overhead and no shared memory
* `asyncio`: single-threaded, cooperative concurrency with `async def`/`await`, `asyncio.gather()`, and top-level `await` inside jupyter cells
* a rule of thumb for picking the right tool: asyncio for lots of I/O-bound tasks, threading for I/O-bound work with blocking libraries, multiprocessing for real CPU-bound speedup